In [1]:
# Add autoreload at the top of the notebook you're working on 
# in order for it to auto refresh when you change the 'project_package'
%load_ext autoreload
%autoreload 2

# Table of contents

[1. Import and align datasets for model training](#section-1)   
&emsp; [a. Load the processed recipe dataset](#section-1a)  
&emsp; [b. Load user rating data](#section-1b)  
&emsp; [c. Create user preference data](#section-1c)

[2. Initialize Chroma vectorstore with document embedding](#section-2)   
&emsp; [a. Create a collection for item data](#section-2a)  
&emsp; [b. Create a collection for user preferences](#section-2b)

[3. Train recommendation models ](#section-3)   
&emsp; [a. Preparing the training/test datasets and metric format](#section-3a)  
&emsp; [b. Cross validation all models for comparison](#section-3b)<br>
&emsp; [c. Retraining chosen models on full dataset](#section-3c)

[4. UI implementation](#section-4)   
&emsp; [4.1. Candidate selection](#section-4.1)  
&emsp; [4.1. Recommendation filtering  & Reranking](#section-4.2)<br>
&emsp;&emsp; [4.2.1. Item-content filtering](#section-4.2.1)<br>
&emsp;&emsp; [4.2.2. Collaboration filtering](#section-4.2.2)<br>
&emsp;&emsp; [4.2.3. Hybrid filtering](#section-4.2.3)



Import support libraries & modules

In [ ]:
import os,io
import warnings
import threadpoolctl
from pathlib import Path
import logging
import ast
logging.getLogger("httpx").setLevel(logging.WARNING)  # hide logging in cell when using chroma vectorstore
from dotenv import load_dotenv

import pandas as pd
import numpy as np
from langchain_community.document_compressors import FlashrankRerank
from flashrank import Ranker
from rank_bm25 import BM25Plus
from rectools.model_selection import cross_validate
from rectools.model_selection.random_split import RandomSplitter
from rectools.model_selection.last_n_split import LastNSplitter
from rectools.models import load_model,ImplicitItemKNNWrapperModel,ImplicitALSWrapperModel,LightFMWrapperModel,ImplicitBPRWrapperModel
from rectools.models.pure_svd import PureSVDModel
import implicit
from implicit.nearest_neighbours import TFIDFRecommender, BM25Recommender
from implicit.als import AlternatingLeastSquares as CPU_AlternatingLeastSquares
from implicit.gpu.als import AlternatingLeastSquares as GPU_AlternatingLeastSquares
from implicit.bpr import BayesianPersonalizedRanking as CPU_BayesianPersonalizedRanking
from implicit.gpu.bpr import BayesianPersonalizedRanking as GPU_BayesianPersonalizedRanking
from lightfm import LightFM

from project_package.data_preprocessing.utils import chroma_filter_operator
from project_package.modeling.recommendation_utils import (
    preprocessing_docs,generate_metric_objs,get_embedding_model,
    VectorstoreLoader,load_vector_store,generate_feature_constraint
    )
from project_package.data_preprocessing.default import (
    USER,ITEM,DOC_TEMPLATE,USER_PROFILE_TEMPLATE,
    RECIPE_COLS,RECIPE_META_COLS,PREFERENCE_COLS
    )
from project_package.data_preprocessing.resource_access import (
    load_preference_data,load_user_review_data,load_recipe_data,construct_rec_train_dataset
    )
from project_package.aws.data_access import pandas_sql_df,build_s3_client
from project_package.aws.model_store import upload_model_file,get_object_bytes

load_dotenv()  # load env variables from .evn
root_directory = Path(os.getcwd()).parent  #NOTE: update of notebook location changed


/home/minhkha/.conda/envs/captone_env/lib/python3.11/site-packages/dask/dataframe/__init__.py:42: FutureWarning: 
Dask dataframe query planning is disabled because dask-expr is not installed.

You can install it with `pip install dask[dataframe]` or `conda install dask`.
This will raise in a future version.

  warnings.warn(msg, FutureWarning)
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/minhkha/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


<a id='section-1'></a>
## 1. Import and align datasets for model training


a. Load the processed recipe dataset
<a id='section-1a'></a>

In [ ]:
recipe_df = load_recipe_data(root_directory)
recipe_df.head(2)

/home/minhkha/MADS-Capstone-food-recommendation-system/project_package/data_collection/utility.py:183: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(f,usecols=usecols) for f in files]


,original_id,recipe_name,instructions,calories,source,prep_time,cook_time,total_time,ingredients,who_score,...,recipe_id,cuisine,cooking_method,difficulty,protein_content,fiber_content,fat_content,carbohydrate_content,sodium_content,s3_key
0,39,Biryani,['Soak saffron in warm milk for 5 minutes and ...,1110.7,foodcom,240.0,25.0,265.0,"[saffron, milk, green chili, onion, garlic, ga...",1,...,2,asian,"[braise, simmer, fry, bake]",advanced,high,high,high,high,medium,recipes/2.json
1,40,Best Lemonade,"['Into a 1 quart Jar with tight fitting lid, p...",311.1,foodcom,30.0,5.0,35.0,"[sugar, lemon zest, water, lemon juice]",3,...,1,unknown,[unknown],intermediate,low,low,low,high,low,recipes/1.json


Retrieve all unique labels of recipe features to be used as dropdown list in the UI

In [3]:
if not os.path.exists(root_directory / 'data/processed/ui_feature_constraints.csv'):
    constraint_df = generate_feature_constraint(
        recipe_df,
        root_directory / "data/processed/ingredient_counts_ingredients_canonical_final_le_5_replace.csv"
    )
    constraint_df.to_csv(root_directory / 'data/processed/ui_feature_constraints.csv',index=False)
else:
    constraint_df = pd.read_csv(root_directory / 'data/processed/ui_feature_constraints.csv')
constraint_df.head(2)

,feature,value,type
0,calories,"[0.0, 4999.8]",numeric
1,prep_time,"[0.0, 1440.0]",numeric


b. Load user rating data
<a id='section-1b'></a>

In [ ]:
user_reviews = load_user_review_data(root_directory,recipe_df)
user_reviews.head(3)

,user_id,recipe_id,rating,modified_time
0,0,14063,4,2002-02-19 12:32:18
1,0,17605,5,2005-02-05 15:06:54
2,0,7593,5,2002-05-02 14:20:30


c. Create user preference data
<a id='section-1c'></a>

In [ ]:
preference_df = load_preference_data(root_directory,recipe_df,user_reviews)
preference_df.head(3)

,user_id,ingredients,cuisine,cooking_method,difficulty,protein_content,fiber_content,fat_content,carbohydrate_content,sodium_content
0,0,"['butter', 'salt']",['american'],['bake'],['intermediate'],['low'],['low'],[],[],['medium']
1,1,"['egg', 'salt', 'sugar']",['american'],['bake'],['intermediate'],['low'],['low'],['low'],['high'],[]
2,2,[],['american'],[],['intermediate'],[],['low'],[],[],[]


## 2. Initialize Chroma vectorstore with document embedding
<a id='section-2'></a>

Load the embedding model

In [6]:
embedding_model = get_embedding_model(
    huggingface_model_path="BAAI/bge-small-en-v1.5",  # NOTE: Change this embedding to foodbert later if necessary
    local_model_name="bge-small",
    device="cuda"
)
chroma_path = root_directory / "data/processed/chroma_db"  #NOTE: Change the Path if necessary

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

a. Create a collection for item data
<a id='section-2a'></a>

In [7]:
#NOTE: Run the cell again when a batch is failed to continue

store_document = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_document:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="recipe_collection",  #NOTE: Change the collection name if necessary,
            embedding = embedding_model,
            doc_template = DOC_TEMPLATE,
            input_data = recipe_df,
            format_cols = RECIPE_COLS,
            meta_cols = RECIPE_META_COLS, # include item ID for later filter tasks
            persist_directory = chroma_path,
            docID_col = ITEM
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    vectorstore = load_vector_store(
        collection_name="recipe_collection",
        embedding_model=embedding_model,
        persist_directory=chroma_path
    )

Starting ingestion from index 0...


100%|██████████| 395/395 [42:31<00:00,  6.46s/it]

Calculation time for embeddings: 2551.07s
Data ingestion completed.


b. Create a collection for user preferences
<a id='section-2b'></a>

In [8]:
#NOTE: Run the cell again when a batch is failed to continue

store_user_pref = False  # Set to True when you need to run the embedding loading again
vector_loader = None

if store_user_pref:
    if vector_loader is None:
        vector_loader = VectorstoreLoader(
            collection_name="user_recipe_preference",  #NOTE: Change the collection name if necessary,
            embedding = embedding_model,
            doc_template = USER_PROFILE_TEMPLATE,
            input_data = preference_df,
            format_cols = PREFERENCE_COLS,
            meta_cols = None,
            persist_directory = chroma_path,
            docID_col = USER
        )

    load_finished =  vector_loader.add_initial_documents(batch_size=1000)
    if load_finished:
        user_vectorstore = vector_loader.return_vectorstore()
        del vector_loader  # to reduce memory usage
else:
    user_vectorstore = load_vector_store(
        collection_name="user_recipe_preference",
        embedding_model=embedding_model,
        persist_directory=chroma_path
    )

Starting ingestion from index 0...


100%|██████████| 13/13 [00:37<00:00,  2.86s/it]

Calculation time for embeddings: 37.16s
Data ingestion completed.


## 3. Train recommendation models 
<a id='section-3'></a>

a. Preparing the training/test datasets and metric format
<a id='section-3a'></a>

In [9]:
k = 10  # number of recommendation to create

# load data to Rectools format
dataset = construct_rec_train_dataset(
    user_reviews,
    recipe_df,
    preference_df,
    use_datetime = True
)

In [10]:
# Retrieve item embeddings from Chromastore
# we are using item embedding instead of onehot coded to compare item similarity
ids = vectorstore._collection.get(include=['embeddings'])['ids']  
doc_embeddings = vectorstore._collection.get(include=['embeddings'])['embeddings']
idx = pd.Index(ids,name='item_id',dtype=int)
embedding_docs = pd.DataFrame(doc_embeddings,index=idx)

# create metric objects for k recommendations
metrics = generate_metric_objs(embedding_docs,k=k)

### List of chosen recommendation models to train

* **ItemKNN model**

This is a wrapper model for item-item nearest neighbour models. Those models are item-content recommendation models, which based purely on the content and doesn't require user-item interaction or user preferences. A few recommendation model belongs to this type are BM25, TFIDF,etc.

* **SVD model**

This is a basic collaboration model where we only take the user-item score interactions then apply a dimension reduction algorithm to create embedding representation in a latent vector space. This allows the model to generate rating for unseen items and create recommendation to the users.

* **AlternatingLeastSquares**

Goal of the model is to present interactions matrix as a product of user(X) and item(Y) embeddings. Implicit ALS model treats all non-zero entries in the matrix as value. The actual weight of the interactions is treated as confidence in the observation. Zero entries receive low confidence since this they are treated as missing values and might actually hide items highly relevant to users. Non-zero entries with high confidence will have greater impact on the loss when not predicted correctly.

* **BayesianPersonalizedRanking**

Bayesian personalized ranking introduces a pairwise loss instead. For each user model takes a pair of items: one positive and one negative where positive item was present in user interactions and negative item wasn’t. The goal of the algorithm is to rank positive item higher then negative one. It is useful for cases when only positive interactions are present in data and when the goal is to maximize ROC AUC.

* **LightFM**

A hybrid latent representation recommender model.

The model learns embeddings (latent representations in a high-dimensional space) for users and items in a way that encodes user preferences over items. When multiplied together, these representations produce scores for every item for a given user; items scored highly are more likely to be interesting to the user.

The user and item representations are expressed in terms of representations of their features: an embedding is estimated for every feature, and these features are then summed together to arrive at representations for users and items


b. Cross validation all models for comparison
<a id='section-3b'></a>

<ins>skip this part if you already train the models</ins>

In [11]:
# For implicit ALS
os.environ["OPENBLAS_NUM_THREADS"] = "1"
threadpoolctl.threadpool_limits(1, "blas")

tfidf_model = ImplicitItemKNNWrapperModel(TFIDFRecommender())
bm25_model = ImplicitItemKNNWrapperModel(BM25Recommender(K1=1.5))
svd_model = PureSVDModel(factors=150,use_gpu=True)

if implicit.gpu.HAS_CUDA:
    als_model = ImplicitALSWrapperModel(GPU_AlternatingLeastSquares(factors=150,random_state=0))
    bpr_model = ImplicitBPRWrapperModel(GPU_BayesianPersonalizedRanking(factors=150,random_state=0))
else:
    als_model = ImplicitALSWrapperModel(CPU_AlternatingLeastSquares(factors=150,random_state=0))
    bpr_model = ImplicitBPRWrapperModel(CPU_BayesianPersonalizedRanking(factors=150,random_state=0))

#NOTE: Update the loss function when computer support thread
lightfm_model = LightFMWrapperModel(LightFM(no_components=100, loss='warp',random_state=0))  

models = {
    "TFIDFRecommender":tfidf_model,
    "BM25Recommender":bm25_model,
    "PureSVDModel":svd_model,
    "AlternatingLeastSquares":als_model,
    "BayesianPersonalizedRanking":bpr_model,
    "LightFM":lightfm_model
}

# splitter = RandomSplitter(test_fold_frac=0.195, random_state=0,n_splits=5)  # test_fold_frac * n_splits can only be close to 100%, otherwise it's impossible to split
splitter = LastNSplitter(n=3,n_splits=5)

/home/minhkha/.conda/envs/captone_env/lib/python3.11/site-packages/rectools/models/pure_svd.py:113: UserWarning: Forced to use CPU. CuPy is not available.
  warnings.warn("Forced to use CPU. CuPy is not available.")


In [12]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    cv_results = cross_validate(
        dataset=dataset,
        splitter=splitter,
        models=models,
        metrics=metrics,
        k=k,
        filter_viewed=True,
    )

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
cross_validate_df = (
    pd.DataFrame(cv_results["metrics"])
    .drop(columns="i_split")
    .groupby(["model"], sort=False)
    .agg(["mean"])
)
cross_validate_df.columns = cross_validate_df.columns.droplevel(1)
cross_validate_df.to_csv(root_directory / 'models/recommendation_model_comparison.csv',index =False)
cross_validate_df

,Recall@10,Precision@10,NDCG@10,Novelty@10,AvgRecPopularity@10,Diversity@10,Serendipity@10
model,,,,,,,
TFIDFRecommender,0.003494,0.003494,0.001191,12.955755,0.000023,0.259640,7.279343e-07
BM25Recommender,0.003020,0.003020,0.000978,12.208594,0.000006,0.257510,1.048686e-06
PureSVDModel,0.017139,0.017139,0.005588,5.458022,0.000535,0.250023,4.400246e-07
AlternatingLeastSquares,0.000060,0.000060,0.000016,12.451164,0.000005,0.240452,1.969505e-08
BayesianPersonalizedRanking,0.005971,0.005971,0.001870,8.605161,0.000088,0.247444,1.171742e-06
LightFM,0.000034,0.000034,0.000020,13.064288,0.000002,0.218530,1.149873e-08


c. Retraining chosen models on full dataset
<a id='section-3c'></a>

In [15]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    svd_model = PureSVDModel(factors=150,use_gpu=True)

    if implicit.gpu.HAS_CUDA:
        als_model = ImplicitALSWrapperModel(GPU_AlternatingLeastSquares(factors=150,random_state=0))
    else:
        als_model = ImplicitALSWrapperModel(CPU_AlternatingLeastSquares(factors=150,random_state=0))
    #NOTE: Update the loss function when computer support thread
    lightfm_model = LightFMWrapperModel(LightFM(no_components=100, loss='warp',random_state=0))  

    svd_model.fit(dataset)
    svd_model.save(root_directory / "models/recommendation_models/svd_recommendation_model.pkl")  #NOTE: Update Path if necessary

    als_model.fit(dataset)
    als_model.save(root_directory / "models/recommendation_models/als_recommendation_model.pkl")  #NOTE: Update Path if necessary

    lightfm_model.fit(dataset)
    lightfm_model.save(root_directory / "models/recommendation_models/lightFM_recommendation_model.pkl")  #NOTE: Update Path if necessary

    #NOTE: We can switch to save model in S3 bucket instead
    # s3_client = build_s3_client(
    #     os.environ['AWS_ACCESS_KEY'],
    #     os.environ['AWS_SECRET_KEY'],
    #     os.environ['AWS_SESSION_TOKEN']
    # )
    # # upload object from path
    # _ = upload_model_file(s3_client,root_directory / "models/recommendation_models/svd_recommendation_model.pkl",'svd_recommendation_model')
    # _ = upload_model_file(s3_client,root_directory / "models/recommendation_models/als_recommendation_model.pkl",'als_recommendation_model')
    # _ = upload_model_file(s3_client,root_directory / "models/recommendation_models/lightFM_recommendation_model.pkl",'lightfm_recommendation_model')

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

<a id='section-4'></a>
## 4. UI implementation


Load the trained recommendation models

In [32]:
#NOTE: Load the models, for big model, we might need to load this from S3 or google drive
svd_model = load_model(root_directory / "models/recommendation_models/svd_recommendation_model.pkl")
als_model = load_model(root_directory / "models/recommendation_models/als_recommendation_model.pkl")
lightfm_model = load_model(root_directory / "models/recommendation_models/lightFM_recommendation_model.pkl")

# can LOAD model from S3 instead if it's there
# s3_client = build_s3_client(
#     os.environ['AWS_ACCESS_KEY'],
#     os.environ['AWS_SECRET_KEY'],
#     os.environ['AWS_SESSION_TOKEN']
# )
# svd_model = load_model(io.BytesIO(get_object_bytes(s3_client,'svd_recommendation_model')))
# als_model = load_model(io.BytesIO(get_object_bytes(s3_client,'als_recommendation_model')))
# lightfm_model = load_model(io.BytesIO(get_object_bytes(s3_client,'lightfm_recommendation_model')))

### 4.1. Candidate selection
<a id='section-4.1'></a>

In [33]:
# create dummy filter component values to mimic UI filters

test_dict = pd.DataFrame(dict(
    filter_type = ["filter_checklist","filter_dropdown","filter_slider","filter_slider"],
    filter_name = ['ingredients','cuisine','total_time','calories'],
    filter_value = [['egg', 'flour', 'butter','salt','sugar','wheat'],
                    ['fusion','american','mediterranean'],
                    [0,120],[0,4000]
                    ],
    priority_type = ['exact','exact','exact','exact']
))

operator_type_mapping = dict(
    filter_name = ["ingredients","cuisine","total_time","calories"],
    # record_type = ['list','string','number','number'],
    operator_type = ['$or:$contains','$in','$range','$range']
)

# map UI feature names to database name if they are different
name_mapping = {
}


In [34]:
filter_operators = chroma_filter_operator(test_dict,operator_type_mapping,name_mapping)

retriever = vectorstore.as_retriever(
    search_kwargs = dict(
        k=1000,  # We retrieve the best 1000 results
        filter=filter_operators
    )
)

test_query = "a meat hamburger"

retrieved_items =  retriever.invoke(test_query)  # retrieve the top search result from the vectorstore that match the query and metadata filter conditions

candidate_ids = [item.id for item in retrieved_items]

result_docs = [item.page_content for item in retrieved_items]

retrieved_items[:5]

[Document(id='81154', metadata={'difficulty': 'intermediate', 'fat_content': 'medium', 'carbohydrate_content': 'low', 'calories': 162.5, 'sodium_content': 'low', 'prep_time': 2.0, 'fsa_score': 1, 'protein_content': 'medium', 'cuisine': 'american', 'total_time': 22.0, 'cooking_method': ['simmer', 'roast'], 'who_score': 1, 'fiber_content': 'low', 'recipe_id': 81154, 'cook_time': 20.0, 'ingredients': ['beef', 'flour', 'water']}, page_content="\nThe recipe name:Meat and Gravy.\nRecipe instruction:\n['Brown hamburger meat in a large skillet.', 'when completely brown, add spoonfuls of flour to soak up the grease.', 'Mixture will look crumbly.', 'Add a glass of water.', 'Use a fork and stir in the water.', 'This will make a very thick gravy.', 'simmer on low for just a minute.', 'Serve over toast.']\nIngredient list:\n['beef', 'flour', 'water']\nCuisine:american\nCooking method:['simmer', 'roast']\nThe difficulty is intermediate.\nProtein content is medium.\nFiber content is low.\nFat content

### 4.2. Recommendation filtering  & Reranking
<a id='section-4.2'></a>

This step is used to further choosing the top recommendations that match the user preferences and query content before re-ranking

#### 4.2.1. Item-content filtering
<a id='section-4.2.1'></a>

We can't use the BM25Recommender model in Rectools for production, because it still need user ratings, but in the UI, we don't actually have user ratings for new users (cold start), so we need to build a pipeline that doesn't utilize user ratings to recommend.

We will be using the BM25Plus model for matching item-contents between the query and the documents

In [35]:
tokenized_corpus = preprocessing_docs(result_docs)
tokenized_query = preprocessing_docs(test_query)

#BM vectorizer model
bm25 = BM25Plus(tokenized_corpus)

doc_scores = bm25.get_scores(tokenized_query)

top_100 = np.array(candidate_ids)[np.argsort(doc_scores)[::-1][:100]]  # sort the similarity score descendingly
top_100_docs = vectorstore.get_by_ids(top_100)
top_100_docs[:5]

[Document(id='890', metadata={'recipe_id': 890, 'cuisine': 'american', 'fat_content': 'high', 'carbohydrate_content': 'high', 'who_score': 0, 'fiber_content': 'high', 'cook_time': 60.0, 'difficulty': 'intermediate', 'calories': 899.9, 'ingredients': ['hamburger', 'bacon', 'rice', 'tomato', 'salt', 'pea', 'cheese'], 'prep_time': 20.0, 'sodium_content': 'high', 'total_time': 80.0, 'fsa_score': 22, 'protein_content': 'high', 'cooking_method': ['bake']}, page_content="\nThe recipe name:Hamburger Casserole.\nRecipe instruction:\n['Cook all meat until browned; drain fat.', 'Place remaining ingredients in alternate  layers in a casserole and top with bread crumbs.', 'Drizzle with 2 teaspoons melted  butter and bake 1 hour at 350°F.']\nIngredient list:\n['hamburger', 'bacon', 'rice', 'tomato', 'salt', 'pea', 'cheese']\nCuisine:american\nCooking method:['bake']\nThe difficulty is intermediate.\nProtein content is high.\nFiber content is high.\nFat content is high.\nCarbohydrate content is high.

Rerank the documents

In [36]:
# Need to remapping the document ID because using FlashrankRerank the document id in the result will be overwriten with index from 0->n
doc_ids_map = dict(
    zip(range(len(top_100)),top_100.tolist())
)

reranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",  # NOTE: Can change to a different Flashrank model of your liking
    cache_dir=os.environ["FLASHRANK_PATH"]
)
compressor = FlashrankRerank(client=reranker,top_n=k)

2026-04-14 22:01:07.008067071 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2749971, index: 0, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-04-14 22:01:07.008131224 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2749973, index: 2, mask: {6, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-04-14 22:01:07.008177268 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2749974, index: 3, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-04-14 22:01:07.008647490 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2749985, index: 14, mask: {32, }, error code: 22 error msg: Invalid argument. Specify the

In [37]:
rerank_result = compressor.compress_documents(
    top_100_docs,
    query = test_query + "and lot's of vegetable "  #NOTE: can add extra context here to the query
)

for doc in rerank_result:  # reranking override the doc id so need to change it bank
    doc.metadata['id'] = doc_ids_map[doc.metadata['id']]

data = []

for doc in rerank_result:   # your list of Document objects
    row = doc.metadata.copy()
    row["page_content"] = doc.page_content
    data.append(row)

pd.DataFrame(data)


,id,relevance_score,cooking_method,sodium_content,total_time,fsa_score,prep_time,fat_content,fiber_content,protein_content,carbohydrate_content,who_score,calories,recipe_id,difficulty,cuisine,cook_time,ingredients,page_content
0,144420,0.924929,[grill],medium,25.0,6,10.0,high,low,medium,low,1,313.7,89108,beginner,american,15.0,"[beef, basil leaf, red chili pepper, olive oil...",\nThe recipe name:Grilled Burgers With Garden ...
1,220750,0.839732,[simmer],high,90.0,0,30.0,low,low,medium,medium,3,153.0,165085,intermediate,american,60.0,"[turkey, onion, celery, rice, salt, potato, ca...",\nThe recipe name:Hamburger Vegetable Soup.\nR...
2,66666,0.555332,"[sautee, boil, simmer, stir_fry]",high,50.0,10,15.0,medium,medium,high,medium,0,318.6,144420,advanced,fusion,35.0,"[hamburger, onion, salt, pepper, celery, water...",\nThe recipe name:Hamburger Chow Mein.\nRecipe...
3,109095,0.539156,"[grill, fry, bake]",high,25.0,18,10.0,high,low,high,medium,1,630.6,426029,intermediate,american,15.0,"[butter, beef, bacon, onion, cheddar cheese, e...",\nThe recipe name:Aussie Style Burger With the...
4,98256,0.487531,"[boil, simmer]",medium,55.0,-2,10.0,low,medium,medium,medium,1,208.4,234941,intermediate,american,45.0,"[tomato, green bean, potato, hamburger meat, o...",\nThe recipe name:Comfort Beef Soup.\nRecipe i...
5,34582,0.395674,"[braise, simmer]",high,80.0,7,20.0,medium,low,medium,medium,0,259.0,468098,intermediate,american,60.0,"[beef, celery, onion, green bell pepper, ketch...",\nThe recipe name:Colorado Loose Hamburgers.\n...
6,427744,0.374333,"[fry, bake]",high,70.0,15,30.0,high,low,medium,medium,0,416.4,43211,intermediate,american,40.0,"[beef, onion, garlic powder, garlic, salt, pep...",\nThe recipe name:Hamburger Pacific.\nRecipe i...
7,177076,0.363915,[grill],high,30.0,22,20.0,high,low,high,medium,0,701.1,131810,beginner,american,10.0,"[milk, salt, black pepper, garlic, cheese]",\nThe recipe name:Well-Done Hamburgers on a Ch...
8,205710,0.359289,[bake],high,50.0,7,30.0,medium,medium,medium,medium,1,304.0,143826,intermediate,american,20.0,"[mashed potato, beef, onion, mushroom, salt, b...",\nThe recipe name:Hamburger Pie.\nRecipe instr...
9,66905,0.227808,"[boil, simmer, fry]",high,45.0,8,15.0,medium,high,medium,high,2,357.0,98256,intermediate,american,30.0,"[beef, onion, garlic, green bell pepper, tomat...","\nThe recipe name:Hamburger, Macaroni, and Bea..."


### 4.2.2. Collaboration filtering
<a id='section-4.2.2'></a>

For this section, we are using ALS model, which relies on user-item rating interaction and item-item interaction to create the recommendations. A drawback of this method is that without any initial ratings for new user (cold items), we can't make a recommendation for them. Therefore, to address this problem. We are going to have user input some initial preferences or try to figure out their short-term preference from their filter choices.

In [38]:
test_user_profile = USER_PROFILE_TEMPLATE.format(*preference_df.iloc[3][PREFERENCE_COLS].tolist())

print(test_user_profile)


Favorite ingredients are: ['garlic'].
Favorite cuisine are: ['american', 'asian'].
Preferred cooking method: ['bake', 'boil', 'sautee', 'simmer'].
Preferred cooking difficulty: ['intermediate'].
Preferred Protein content is ['high', 'medium'].
Preferred Fiber content is ['low'].
Preferred Fat content is ['high', 'medium'].
Preferred Carbohydrate content is [].
Preferred Sodium content is ['high'].



We try to retrieve the top user profiles that similar to the user

In [39]:
user_retriever = user_vectorstore.as_retriever(
    search_kwargs = dict(
        k=10,  # We retrieve the 10 best similar users
    )
)

retrieved_users =  user_retriever.invoke(test_user_profile)  # retrieve the top search result from the vectorstore that match the query and metadata filter conditions

match_user_ids = np.array([item.id for item in retrieved_users],dtype=int)

retrieved_users[:5]

[Document(id='3', metadata={}, page_content="\nFavorite ingredients are: ['garlic'].\nFavorite cuisine are: ['american', 'asian'].\nPreferred cooking method: ['bake', 'boil', 'sautee', 'simmer'].\nPreferred cooking difficulty: ['intermediate'].\nPreferred Protein content is ['high', 'medium'].\nPreferred Fiber content is ['low'].\nPreferred Fat content is ['high', 'medium'].\nPreferred Carbohydrate content is [].\nPreferred Sodium content is ['high'].\n"),
 Document(id='5419', metadata={}, page_content="\nFavorite ingredients are: ['garlic', 'onion'].\nFavorite cuisine are: ['american', 'asian'].\nPreferred cooking method: ['bake', 'boil', 'sautee', 'simmer'].\nPreferred cooking difficulty: ['intermediate'].\nPreferred Protein content is [].\nPreferred Fiber content is ['low'].\nPreferred Fat content is ['high'].\nPreferred Carbohydrate content is ['high'].\nPreferred Sodium content is ['high'].\n"),
 Document(id='3169', metadata={}, page_content="\nFavorite ingredients are: ['garlic',

We find the top recommendations for the top users that are similar to the test user

In [40]:
model_recommendations = als_model.recommend(
    match_user_ids,
    dataset,
    k=k,
    filter_viewed=False
)
model_recommendations.head(3)

,user_id,item_id,score,rank
0,3,80210,131.601624,1
1,3,5593,5.294956,2
2,3,402279,5.099116,3


Finally we us a weighted random selection of recommendations from the pool of recommendations for similar users

In [41]:
# The strategy is to count how many time a recommendation has been suggested for each user, then we have a weighted shuffle
# to select the k recommendations

counts = model_recommendations.item_id.value_counts()

# Exact IDs and the weights
items = counts.index.values
weights = counts.values

# Normalize the weights
probabilities = weights / weights.sum()

np.random.seed(0)  # can turn seed on/off

recommend_ids = np.random.choice(
    items, 
    size=k, 
    replace=False, 
    p=probabilities
)

recommend_ids

array([325166, 167497, 446234, 186695, 279561,  80210, 277029, 369319,
       145639, 237019])

### 4.2.3. Hybrid filtering
<a id='section-4.2.3'></a>

With normal collaboration models, many of them won't be able to handle cold-start for items without ratings or user without any reviews. Hybrid models are developed to address this problem, one of them is LightFM.

In [42]:
# we are using the same test query
print(test_query)
# and candidate retrieved from vectorstore
candidate_ids[:5]

a meat hamburger


['81154', '141462', '390441', '5114', '427391']

Using the list of candidate, we input that to the lightFM model to rank the top 100 recommendations for each similar users. Note that because the user preference has been taken into context, the top recommendations in the step might have derived from the text_query

In [43]:
recommendations = lightfm_model.recommend(
    users=match_user_ids,  # we also reuse similar users as we don't have rating for new user yet
    dataset=dataset,
    k=100,
    items_to_recommend=np.array(candidate_ids,dtype=int), # Can contain either hot or warm items
    filter_viewed = False
)

In [44]:
# them we sum the score of each recommended items and sort them
top_100 = recommendations.groupby('item_id')['score'].sum().sort_values(ascending=False).index[:100].to_numpy()
top_100[:5]
top_100_docs = vectorstore.get_by_ids(top_100.astype(str))
top_100_docs[:5]

[Document(id='1204', metadata={'fsa_score': 23, 'cook_time': 30.0, 'cooking_method': ['grill', 'boil', 'simmer', 'bake'], 'fat_content': 'high', 'difficulty': 'intermediate', 'fiber_content': 'high', 'prep_time': 0.0, 'protein_content': 'low', 'who_score': 0, 'recipe_id': 1204, 'cuisine': 'american', 'calories': 495.9, 'sodium_content': 'high', 'ingredients': ['beef burger', 'onion', 'butter', 'flour', 'worcestershire sauce', 'wine vinegar', 'tomato paste', 'canned tomato'], 'carbohydrate_content': 'high', 'total_time': 30.0}, page_content="\nThe recipe name:Beef Burgers With Orange Barbecued Sauce.\nRecipe instruction:\n['Brown the burgers and saute the onion gently in the oil.', 'Place in a casserole.  Make a roux with the butter and flour, add the orange juice, rind, Worcestershire sauce, vinegar and tomato paste.  Cook well, stir in the tomatoes and season well.', 'Pour into the casserole and cook at 160 C 30 - 40 minutes.', 'Taste and adjust seasonings before serving.', 'Serve wit

Then the final step would be to rerank the list recommendation from previous step using the text_query context

In [48]:
# Need to remapping the document ID because using FlashrankRerank the document id in the result will be overwriten with index from 0->n
doc_ids_map = dict(
    zip(range(len(top_100)),top_100.tolist())
)

reranker = Ranker(
    model_name="ms-marco-MiniLM-L-12-v2",  # NOTE: Can change to a different Flashrank model of your liking
    cache_dir=os.environ["FLASHRANK_PATH"]
)
compressor = FlashrankRerank(client=reranker,top_n=k)

2026-04-14 22:03:01.202940540 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2750356, index: 0, mask: {4, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-04-14 22:03:01.202988257 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2750358, index: 2, mask: {6, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-04-14 22:03:01.203024095 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2750359, index: 3, mask: {2, }, error code: 22 error msg: Invalid argument. Specify the number of threads explicitly so the affinity is not set.
2026-04-14 22:03:01.203303513 [E:onnxruntime:Default, env.cc:227 ThreadMain] pthread_setaffinity_np failed for thread: 2750370, index: 14, mask: {32, }, error code: 22 error msg: Invalid argument. Specify the

In [49]:
rerank_result = compressor.compress_documents(
    top_100_docs,
    query = test_query #NOTE: can add extra condition here to the query
)

for doc in rerank_result:  # reranking override the doc id so need to change it bank
    doc.metadata['id'] = doc_ids_map[doc.metadata['id']]

data = []

for doc in rerank_result:   # your list of Document objects
    row = doc.metadata.copy()
    row["page_content"] = doc.page_content
    data.append(row)

pd.DataFrame(data)


,id,relevance_score,fiber_content,cook_time,ingredients,carbohydrate_content,total_time,calories,sodium_content,fsa_score,protein_content,cooking_method,prep_time,cuisine,difficulty,recipe_id,fat_content,who_score,page_content
0,304183,0.965417,low,20.0,"[beef, egg, barbecue sauce, ketchup, garlic po...",low,30.0,297.8,medium,7,medium,[grill],10.0,american,beginner,380759,high,1,\nThe recipe name:BBQ Hamburgers.\nRecipe inst...
1,169678,0.954316,low,20.0,"[beef, egg, green onion, oregano leaf, basil l...",low,40.0,317.5,medium,9,medium,"[grill, bake]",20.0,american,intermediate,169180,high,1,\nThe recipe name:Chris's Meat Hamburger Patti...
2,160930,0.931702,low,10.0,"[egg, dry mustard, chili sauce, worcestershire...",low,25.0,323.6,high,9,high,[grill],15.0,american,beginner,325868,medium,1,\nThe recipe name:The Burger.\nRecipe instruct...
3,188146,0.913633,low,30.0,"[beef, onion, garlic, oatmeal, milk, egg, whit...",low,40.0,407.1,medium,8,high,"[sautee, boil, simmer, fry]",10.0,american,intermediate,258176,high,1,\nThe recipe name:Amazing Hamburger Steak With...
4,318650,0.912922,low,10.0,"[egg, chili sauce, worcestershire sauce, salt,...",medium,20.0,349.1,high,11,medium,[grill],10.0,american,beginner,266274,medium,0,\nThe recipe name:The Perfect Hamburgers.\nRec...
5,394522,0.911400,low,35.0,"[hamburger meat, onion, olive oil, worcestersh...",low,50.0,598.3,high,17,high,"[sautee, grill, boil, simmer]",15.0,american,advanced,374969,high,1,\nThe recipe name:Hamburger Steak With Onion ...
6,45110,0.905945,medium,45.0,"[tomato, green bean, potato, hamburger meat, o...",medium,55.0,208.4,medium,-2,medium,"[boil, simmer]",10.0,american,intermediate,234941,low,1,\nThe recipe name:Comfort Beef Soup.\nRecipe i...
7,144857,0.878693,medium,20.0,"[hamburger meat, onion, green bell pepper, tom...",high,35.0,625.4,high,18,high,"[simmer, bake, fry]",15.0,american,intermediate,80039,high,0,\nThe recipe name:Hamburger Pie.\nRecipe instr...
8,160818,0.873306,medium,10.0,"[egg, garlic, unsalted butter, parsley, worces...",high,40.0,860.7,high,25,high,"[grill, sautee, boil]",30.0,american,intermediate,420413,high,0,\nThe recipe name:Meatloaf Burger.\nRecipe ins...
9,466836,0.864329,low,25.0,"[beef, onion, salt, catsup, chili sauce, brown...",low,45.0,824.9,high,25,medium,"[grill, boil, simmer, bake]",20.0,american,intermediate,5126,high,0,\nThe recipe name:Barbecue Hamburger Patties.\...
